In [0]:
#RAW TO BRONZE
import yaml
from pyspark.sql.types import NumericType, StringType, DateType, TimestampType, DecimalType, IntegerType
from pyspark.sql.functions import col, when, lit, regexp_replace, date_format, from_utc_timestamp, current_timestamp, year, month, dayofmonth

dbutils.widgets.text("NB_Storage_Account","")
storage_account = dbutils.widgets.get("NB_Storage_Account")

config_path = f"abfss://config@{storage_account}.dfs.core.windows.net/databricks_config.yaml"

config = dbutils.fs.head(config_path)
config_parse = yaml.safe_load(config)

dbutils.widgets.text("NB_Year", "")
dbutils.widgets.text("NB_Month", "")
param_year = str(dbutils.widgets.get("NB_Year"))
param_month = str(dbutils.widgets.get("NB_Month"))
print(config_path)
print(param_year)
print(param_month)

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except:
        return False

for table_name in config_parse["tables"]:
  
        rawpath = f"abfss://raw@{storage_account}.dfs.core.windows.net/sales/{table_name}/Year={param_year}/Month={param_month}/"
        

        if not path_exists(rawpath):
                print(f"SKIPPED: {table_name} Year={param_year} Month={param_month} Folder not found")
                continue
        
        df_to_bronze = spark.read.option("mergeSchema","true").parquet(rawpath)        
        df_to_bronze = df_to_bronze.withColumn("Year",  date_format(col("txdate"), "yyyy")) \
                .withColumn("Month", date_format(col("txdate"), "MM")) \
                .withColumn("Day", date_format(col("txdate"), "dd"))


        if spark.catalog.tableExists(f"<catalog>.bronze.{table_name}"):
                df_to_bronze.write \
                        .format("delta") \
                        .mode("overwrite") \
                        .option("mergeSchema", "true") \
                        .option("replaceWhere", f"Year = '{param_year}' AND Month = '{param_month}'") \
                        .saveAsTable(f"<catalog>.bronze.{table_name}")
                print(f"Saved Bronze Data For: {table_name}, Year={param_year}, Month={param_month}")
        else:
                df_to_bronze.write \
                        .format("delta") \
                        .option("mergeSchema", "true") \
                        .option("path", f"abfss://bronze@{storage_account}.dfs.core.windows.net/sales/{table_name}/") \
                        .partitionBy("Year", "Month", "Day") \
                        .saveAsTable(f"<catalog>.bronze.{table_name}")
        
                print(f"Saved Bronze Data For: {table_name}, Year={param_year}, Month={param_month}")



  